# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example for loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name : {metadata.name}")
print(f"Description : {metadata.description}")

## 2. Data Overview
Review available record sets, field `@id`s, and their columns.

We use entity `@id`s for referencing all record sets and fields as recommended.

In [ ]:
# List available record sets, their @id, and fields @ids

record_sets = dataset.record_sets  # This returns a list of CroissantRecordSet objects

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs.name} (ID: {rs.id})")
    fields = getattr(rs, 'fields', [])
    if fields:  # Can be None if no fields
        for f in fields:
            print(f"  Field: {f.name} (ID: {f.id}, DataType: {f.data_type})")
    else:
        print("  [No fields defined]")
    print()

## 3. Data Extraction
Load data from the main record set(s) into Pandas DataFrames using their `@id`s.

Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets using their @id

dataframes = dict()
for rs in record_sets:
    print(f"Loading records for RecordSet: {rs.name} (ID: {rs.id}) ...")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    print()

# As this dataset has a main data table, display the first few rows of the main record set
if record_sets:
    main_rs = record_sets[0]
    main_rs_id = main_rs.id
    print(f"Columns for main RecordSet (id={main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, categorizing, and grouping.

Make sure to refer to field names via their `@id` from previous cells.

In [ ]:
# For demonstration, select the first numeric field from the main record set (if available)
main_rs = record_sets[0]
main_rs_id = main_rs.id
df = dataframes[main_rs_id]

# Find a numeric field by checking CroissantField.data_type
numeric_field_id = None

for f in getattr(main_rs, 'fields', []):
    # Candidate types: 'Integer', 'Float', 'Number'
    if f.data_type and any(x in f.data_type for x in ['Integer', 'Float', 'Number']):
        numeric_field_id = f.id
        print(f"Using numeric field: {f.name} (ID: {f.id})")
        break

if numeric_field_id is None:
    print("No numeric field found in main RecordSet.")
else:
    # filter threshold:
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (records: {filtered_df.shape[0]}):")
        print(filtered_df.head())

        # normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # grouping by the first categorical field if available
        group_field_id = None
        for f in getattr(main_rs, 'fields', []):
            # Use Text fields or Boolean
            if f.data_type and (('Text' in f.data_type) or ('Boolean' in f.data_type)):
                group_field_id = f.id
                print(f"\nGrouping by categorical field: {f.name} (ID: {f.id})")
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print(f"Numeric field {numeric_field_id} not in DataFrame columns.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the selected numeric field (if available)
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a categorical field was used for grouping, plot boxplot
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, overview, extract, explore, and visualize a Croissant-standard tabular biomedical dataset using the `mlcroissant` Python library.

- All entities (record sets, fields) were referenced by their `@id` fields.
- We explored available fields, loaded the full dataset, and applied numeric filtering, normalization, and grouping.
- Example plots visualize the distribution of selected numeric variables and their relationship with a (categorical) group.

Feel free to further customize analyses or visualizations for your own use-cases and research questions.